# Hospital Crosswalk: OSHA Establishments to CMS Hospitals

Links hospitals that report injuries to OSHA with the same hospitals in CMS data, so that staff injury rates can be set beside hospital quality measures. The method is the one used for nursing homes in [`crosswalk/`](../crosswalk/): blocking on state and ZIP code or on state, city, and house number; fuzzy name and address scores; three confidence tiers; one-to-one assignment.

**Data** OSHA Form 300A, 2025 (submissions through 2026-08-31), hospitals (NAICS 622); CMS Hospital General Information, Complications and Deaths, and Hospital-Acquired Condition Reduction Program files (current releases); facility rates from [`injury-rates/`](../injury-rates/)
**Source data** US Government work, 17 U.S.C. §105

### Published identifiers only

The crosswalk file holds OSHA establishment ID, CMS Certification Number, tier, and match scores. Names and addresses are used for matching and are not published. A sample of matched names for checking is written to `data/` (not published).

### Analysis

Section 5 sets staff handling-injury rates beside hospital-acquired pressure injury (PSI-03), in-hospital falls with hip fracture (PSI-08), and HACRP payment reduction. It is descriptive. Results are reported whichever way they fall.

## 1. Environment

In [1]:
!pip install -q duckdb pandas requests rapidfuzz scipy
import os, re, requests, duckdb, pandas as pd, numpy as np
from rapidfuzz import fuzz
from scipy import stats
pd.set_option('display.max_columns', None); pd.set_option('display.width', 200)
os.makedirs('data', exist_ok=True); os.makedirs('derived', exist_ok=True)
HDRS = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 '
                      '(KHTML, like Gecko) Chrome/120.0 Safari/537.36'}
def fetch(url, path):
    if not os.path.exists(path):
        with requests.get(url, headers=HDRS, stream=True, timeout=1800) as r:
            r.raise_for_status()
            with open(path, 'wb') as f:
                for ch in r.iter_content(1 << 20): f.write(ch)
    print(f'{path}: {os.path.getsize(path)/1e6:.0f} MB')
con = duckdb.connect(':memory:')

## 2. OSHA hospital establishments (Form 300A)

In [2]:
fetch('https://www.osha.gov/sites/default/largefiles/ITA_300A_Summary_Data_2025_through_08-31-2026.csv', 'data/300a.csv')
con.execute("CREATE TABLE s AS SELECT * FROM read_csv_auto('data/300a.csv', all_varchar=true, ignore_errors=true, sample_size=-1)")
cols = {c.lower(): c for c in [r[0] for r in con.execute('DESCRIBE s').fetchall()]}
oc = lambda *n: next(cols[x.lower()] for x in n if x.lower() in cols)
osha = con.execute(f'''
SELECT "{oc('establishment_id')}" AS establishment_id, ANY_VALUE("{oc('establishment_name')}") AS establishment_name,
       ANY_VALUE("{oc('street_address')}") AS street_address, ANY_VALUE("{oc('city')}") AS city,
       ANY_VALUE("{oc('state')}") AS state, ANY_VALUE("{oc('zip_code', 'zip')}") AS zip_code,
       MAX(TRY_CAST("{oc('annual_average_employees')}" AS INT)) AS employees
FROM s WHERE "{oc('naics_code')}" LIKE '622%' GROUP BY 1''').df()
print(f'{len(osha):,} OSHA hospital establishments (NAICS 622)')

data/300a.csv: 88 MB


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

6,608 OSHA hospital establishments (NAICS 622)


## 3. CMS hospitals

In [3]:
catalog = requests.get('https://data.cms.gov/provider-data/data.json', timeout=120).json()['dataset']
def cms_csv(title):
    d = next(d for d in catalog if d['title'].strip().lower() == title.lower())
    url = next(x['downloadURL'] for x in d['distribution'] if x.get('downloadURL', '').endswith('.csv'))
    print(f"{d['title']} | modified {d.get('modified')}"); return url, d.get('modified')
sources = {}
gen_url, sources['Hospital General Information'] = cms_csv('Hospital General Information')
comp_url, sources['Complications and Deaths - Hospital'] = cms_csv('Complications and Deaths - Hospital')
hac_url, sources['Hospital-Acquired Condition (HAC) Reduction Program'] = cms_csv('Hospital-Acquired Condition (HAC) Reduction Program')
pd.Series(sources, name='modified').to_csv('derived/cms-source-files.csv')
gen = pd.read_csv(gen_url, dtype=str)
gen = gen.rename(columns={'Facility ID': 'ccn', 'Facility Name': 'name', 'Address': 'address', 'City/Town': 'city',
                          'State': 'state', 'ZIP Code': 'zip', 'Hospital Type': 'hospital_type'})
print(f'{len(gen):,} CMS hospitals'); gen.hospital_type.value_counts()

Hospital General Information | modified 2026-10-26
Complications and Deaths - Hospital | modified 2026-10-20
Hospital-Acquired Condition (HAC) Reduction Program | modified 2026-01-26
5,435 CMS hospitals


,count
hospital_type,
Acute Care Hospitals,3094
Critical Access Hospitals,1386
Psychiatric,632
Acute Care - Veterans Administration,132
Childrens,94
Rural Emergency Hospital,49
Acute Care - Department of Defense,32
Dedicated Cancer Care,11
Long-term,5


## 4. Matching

In [4]:
import re, numpy as np, pandas as pd
from rapidfuzz import fuzz

SUFFIX = {'street':'st','st.':'st','avenue':'ave','av':'ave','ave.':'ave','road':'rd','rd.':'rd','drive':'dr','dr.':'dr',
          'boulevard':'blvd','blvd.':'blvd','lane':'ln','court':'ct','place':'pl','parkway':'pkwy','highway':'hwy',
          'hiway':'hwy','circle':'cir','terrace':'ter','way':'way','square':'sq','turnpike':'tpke','route':'rt','rte':'rt',
          'north':'n','south':'s','east':'e','west':'w','northeast':'ne','northwest':'nw','southeast':'se','southwest':'sw',
          'suite':'','ste':'','unit':'','building':'','bldg':'','floor':'','fl':''}
NAME_STOP = {'the','of','and','at','a','llc','inc','corp','corporation','co','company','lp','llp','pllc','dba','d','b',
             'center','centre','ctr','hospital','hospitals','hosp','medical','med','health','healthcare','care','system',
             'systems','services','service','facility','campus','operating','operations','for','inpatient','acute',
             'general','community','regional'}

def norm_zip(z):
    z = re.sub(r'\D', '', str(z or ''))
    return z[:5].zfill(5) if z else ''

def norm_addr(a):
    a = str(a or '').lower()
    a = re.sub(r'[#,\.]', ' ', a)
    a = re.sub(r'\b(suite|ste|unit|bldg|building|floor|fl|room|rm)\s*\w+\b', ' ', a)
    toks = [SUFFIX.get(t, t) for t in a.split()]
    return ' '.join(t for t in toks if t)

def house_no(a):
    m = re.match(r'\s*(\d+)', norm_addr(a))
    return m.group(1) if m else ''

def norm_name(n):
    n = str(n or '').lower().replace('&', ' and ')
    n = re.sub(r"[^a-z0-9 ]", ' ', n)
    toks = [t for t in n.split() if t not in NAME_STOP]
    return ' '.join(toks)

def prep(df, id_, name, addr, city, state, zip_):
    out = pd.DataFrame({'id': df[id_].astype(str).str.strip(),
                        'name_raw': df[name], 'addr_raw': df[addr],
                        'city': df[city].astype(str).str.lower().str.strip(),
                        'state': df[state].astype(str).str.upper().str.strip(),
                        'zip5': df[zip_].map(norm_zip)})
    out['name'] = out.name_raw.map(norm_name)
    out['addr'] = out.addr_raw.map(norm_addr)
    out['hno'] = out.addr_raw.map(house_no)
    return out.drop_duplicates('id')

def score_pairs(o, c):
    """o: OSHA establishments, c: CMS facilities (both from prep). Two blocking passes:
    same state + ZIP; and same state + city + house number (catches ZIP typos)."""
    keep = ['id', 'name', 'addr', 'hno', 'city', 'zip5', 'state']
    o, c = o[keep], c[keep]
    b1 = o.merge(c, on=['state', 'zip5'], suffixes=('_o', '_c'))
    b2 = o[o.hno != ''].merge(c[c.hno != ''], on=['state', 'city', 'hno'], suffixes=('_o', '_c'))
    b2 = b2.assign(hno_o=b2.hno, hno_c=b2.hno, city_o=b2.city, city_c=b2.city).drop(columns=['hno', 'city'])
    b1 = b1.assign(zip5_o=b1.zip5, zip5_c=b1.zip5).drop(columns=['zip5'])
    p = pd.concat([b1, b2], ignore_index=True).drop_duplicates(['id_o', 'id_c']).reset_index(drop=True)
    p['name_score'] = [fuzz.token_set_ratio(a, b) / 100 if a and b else 0.0 for a, b in zip(p.name_o, p.name_c)]
    p['addr_score'] = [fuzz.token_sort_ratio(a, b) / 100 if a and b else 0.0 for a, b in zip(p.addr_o, p.addr_c)]
    p['hno_match'] = (p.hno_o == p.hno_c) & (p.hno_o != '')
    p['zip_match'] = p.zip5_o == p.zip5_c
    return p

def tier(r):
    if r.hno_match and r.addr_score >= 0.85 and r.name_score >= 0.60: return 'A'   # same building, name agrees
    if r.hno_match and r.addr_score >= 0.85:                          return 'B'   # same building, name differs (renamed / chain)
    if r.name_score >= 0.90 and r.addr_score >= 0.60:                 return 'B'   # same name, address written differently
    if r.name_score >= 0.95:                                          return 'C'   # same name, same zip, address unclear
    return None

def resolve(p):
    p = p.copy()
    p['tier'] = p.apply(tier, axis=1)
    p = p[p.tier.notna()]
    rank = {'A': 3, 'B': 2, 'C': 1}
    p['total'] = p.tier.map(rank) * 10 + p.name_score + p.addr_score
    p = p.sort_values('total', ascending=False)
    used_o, used_c, keep = set(), set(), []
    for r in p.itertuples():                       # greedy one-to-one
        if r.id_o in used_o or r.id_c in used_c: continue
        used_o.add(r.id_o); used_c.add(r.id_c); keep.append(r.Index)
    return p.loc[keep]

In [5]:
o = prep(osha, 'establishment_id', 'establishment_name', 'street_address', 'city', 'state', 'zip_code')
c = prep(gen, 'ccn', 'name', 'address', 'city', 'state', 'zip')
pairs = score_pairs(o, c); xwalk = resolve(pairs)
print(f'{len(o):,} OSHA establishments, {len(c):,} CMS hospitals, {len(pairs):,} candidate pairs -> {len(xwalk):,} matches')
print(f'OSHA matched: {len(xwalk)/len(o):.1%} | CMS matched: {len(xwalk)/len(c):.1%}')
xwalk.tier.value_counts().sort_index()

6,608 OSHA establishments, 5,435 CMS hospitals, 7,198 candidate pairs -> 3,180 matches
OSHA matched: 48.1% | CMS matched: 58.5%


,count
tier,
A,2536
B,539
C,105


### Match quality

Critical access hospitals are small by law (25 inpatient beds or fewer). If matches are right, OSHA employee counts for matched critical access hospitals should be far below those for matched acute care hospitals, in every tier.

In [6]:
q = (xwalk[['id_o', 'id_c', 'tier']].merge(osha[['establishment_id', 'employees']], left_on='id_o', right_on='establishment_id')
     .merge(gen[['ccn', 'hospital_type']], left_on='id_c', right_on='ccn'))
quality = q.groupby(['tier', 'hospital_type']).employees.agg(['count', 'median']).round(0)
quality.to_csv('derived/match-quality-by-type.csv'); quality

count  median
tier hospital_type                                      
A    Acute Care Hospitals                   1689  1059.0
     Childrens                                40  2165.0
     Critical Access Hospitals               566   196.0
     Dedicated Cancer Care                     3  9781.0
     Long-term                                 1   218.0
     Psychiatric                             233   226.0
     Rural Emergency Hospital                  4    55.0
B    Acute Care Hospitals                    362  1092.0
     Childrens                                11  3740.0
     Critical Access Hospitals               127   178.0
     Dedicated Cancer Care                     1  1437.0
     Long-term                                 1   266.0
     Psychiatric                              36   234.0
     Rural Emergency Hospital                  1   170.0
C    Acute Care - Veterans Administration      1    43.0
     Acute Care Hospitals                     59  1091.0
     Childrens                                 3  3445.0
     Critical Access Hospitals                34   186.0
     Psychiatric                               7   235.0
     Rural Emergency Hospital                  1   220.0

In [7]:
# Names for checking by eye: written to data/, which is not published.
named = xwalk.assign(name_o=xwalk.id_o.map(o.set_index('id').name_raw), name_c=xwalk.id_c.map(c.set_index('id').name_raw))
pd.concat([g.sample(min(30, len(g)), random_state=0) for _, g in named.groupby('tier')])[
    ['tier', 'name_score', 'addr_score', 'name_o', 'name_c']].to_csv('data/spot-check-names.csv', index=False)
out = (xwalk[['id_o', 'id_c', 'tier', 'name_score', 'addr_score', 'hno_match', 'zip_match']]
       .rename(columns={'id_o': 'osha_establishment_id', 'id_c': 'cms_ccn'}).sort_values(['tier', 'cms_ccn']))
out['name_score'] = out.name_score.round(3); out['addr_score'] = out.addr_score.round(3)
out.to_csv('osha-cms-crosswalk-hospital-2025.csv', index=False)
pd.Series({'osha_establishments': len(o), 'cms_hospitals': len(c), 'matches': len(xwalk),
           **{f'tier_{k}': int(v) for k, v in xwalk.tier.value_counts().items()}}).rename('value').to_csv('derived/match-summary.csv')
print(len(out), 'rows written; spot-check names in data/spot-check-names.csv')

3180 rows written; spot-check names in data/spot-check-names.csv


## 5. Staff handling-injury rates beside hospital quality measures

Tiers A and B; hospitals in the injury-rate population of `injury-rates/` (100+ employees, plausible hours, case detail submitted).

In [8]:
FR = 'https://raw.githubusercontent.com/yuxuan-huang-data/us-care-safety-signals/main/injury-rates/derived/facility-rates-2025.csv'
fr = pd.read_csv(FR, dtype={'establishment_id': str})
fr = fr[fr.setting == 'hospital'].drop(columns=['cms_ccn', 'tier'], errors='ignore')
fr['lateral_rate'] = fr.lateral * 200000 / fr.hours
comp = pd.read_csv(comp_url, dtype=str)
psi = comp[comp['Measure ID'].isin(['PSI_03', 'PSI_08'])].assign(score=lambda d: pd.to_numeric(d.Score, errors='coerce'))
psi = psi.pivot_table(index='Facility ID', columns='Measure ID', values='score').reset_index().rename(columns={'Facility ID': 'ccn', 'PSI_03': 'psi03', 'PSI_08': 'psi08'})
hac = pd.read_csv(hac_url, dtype=str)[['Facility ID', 'Payment Reduction']].rename(columns={'Facility ID': 'ccn', 'Payment Reduction': 'hacrp_penalty'})
a = (out[out.tier.isin(['A', 'B'])].merge(fr, left_on='osha_establishment_id', right_on='establishment_id')
     .merge(psi, left_on='cms_ccn', right_on='ccn', how='left').merge(hac, on='ccn', how='left'))
print(len(a), 'linked hospitals with injury rates')
rows = []
for x, y in [('handling_rate', 'psi03'), ('handling_rate', 'psi08'), ('lateral_rate', 'psi03'), ('all_cases_rate', 'psi03')]:
    d = a[[x, y]].dropna()
    rows.append({'staff measure': x, 'patient measure': y, 'hospitals': len(d),
                 'spearman rho': round(stats.spearmanr(d[x], d[y]).statistic, 3), 'p': round(stats.spearmanr(d[x], d[y]).pvalue, 4)})
corr = pd.DataFrame(rows); corr.to_csv('derived/staff-vs-patient-measures.csv', index=False); corr

2458 linked hospitals with injury rates


,staff measure,patient measure,hospitals,spearman rho,p
0,handling_rate,psi03,1697,-0.066,0.0066
1,handling_rate,psi08,1706,-0.062,0.0110
2,lateral_rate,psi03,1697,-0.003,0.9122
3,all_cases_rate,psi03,1697,-0.074,0.0024


In [9]:
pen = a[a.hacrp_penalty.isin(['Yes', 'No'])]
g = pen.groupby('hacrp_penalty').agg(hospitals=('handling_rate', 'size'), median_handling_rate=('handling_rate', 'median'),
                                     median_all_cases_rate=('all_cases_rate', 'median'))
u = stats.mannwhitneyu(pen[pen.hacrp_penalty == 'Yes'].handling_rate, pen[pen.hacrp_penalty == 'No'].handling_rate)
g['mann_whitney_p_handling'] = float(f'{u.pvalue:.2g}')
g.to_csv('derived/handling-rate-by-hacrp-penalty.csv'); g

,hospitals,median_handling_rate,median_all_cases_rate,mann_whitney_p_handling
hacrp_penalty,,,,
No,1300,0.5245,5.1200,0.014
Yes,374,0.4245,4.5915,0.014


### Sensitivity: tier A only

Tier B includes same-address matches whose names differ, which at hospitals can be a different facility at the same campus (for example, a long-term acute care hospital inside a general hospital). Repeating Section 5 on tier A alone shows whether those matches change the result.

In [10]:
aA = a[a.tier == 'A']; rows = []
for x, y in [('handling_rate', 'psi03'), ('handling_rate', 'psi08'), ('lateral_rate', 'psi03'), ('all_cases_rate', 'psi03')]:
    d = aA[[x, y]].dropna(); sr = stats.spearmanr(d[x], d[y])
    rows.append({'staff measure': x, 'patient measure': y, 'hospitals': len(d), 'spearman rho': round(sr.statistic, 3), 'p': round(sr.pvalue, 4)})
penA = aA[aA.hacrp_penalty.isin(['Yes', 'No'])]
gA = penA.groupby('hacrp_penalty').agg(hospitals=('handling_rate', 'size'), median_handling_rate=('handling_rate', 'median'))
gA['mann_whitney_p_handling'] = float(f"{stats.mannwhitneyu(penA[penA.hacrp_penalty == 'Yes'].handling_rate, penA[penA.hacrp_penalty == 'No'].handling_rate).pvalue:.2g}")
corrA = pd.DataFrame(rows); corrA.to_csv('derived/staff-vs-patient-measures-tier-a.csv', index=False)
gA.to_csv('derived/handling-rate-by-hacrp-penalty-tier-a.csv'); print(corrA.to_string(index=False)); gA

 staff measure patient measure  hospitals  spearman rho      p
 handling_rate           psi03       1412        -0.076 0.0041
 handling_rate           psi08       1420        -0.051 0.0534
  lateral_rate           psi03       1412        -0.002 0.9543
all_cases_rate           psi03       1412        -0.085 0.0013


,hospitals,median_handling_rate,mann_whitney_p_handling
hacrp_penalty,,,
No,1088,0.5245,0.007
Yes,303,0.4180,0.007


  adding: derived/ (stored 0%)
  adding: derived/staff-vs-patient-measures.csv (deflated 41%)
  adding: derived/handling-rate-by-hacrp-penalty.csv (deflated 24%)
  adding: derived/cms-source-files.csv (deflated 23%)
  adding: derived/handling-rate-by-hacrp-penalty-tier-a.csv (deflated 16%)
  adding: derived/match-quality-by-type.csv (deflated 55%)
  adding: derived/staff-vs-patient-measures-tier-a.csv (deflated 41%)
  adding: derived/match-summary.csv (deflated 14%)
  adding: osha-cms-crosswalk-hospital-2025.csv (deflated 76%)
  adding: data/spot-check-names.csv (deflated 60%)


## Limitations

- Matching is by name and address; tier C matches are the least certain and are excluded from Section 5.
- A hospital may report to OSHA as several establishments or as one; one-to-one assignment keeps the best match only.
- Section 5 compares establishments, not patients and staff in the same events. An association, or its absence, does not show cause.
- No hospital is named in any published file.